In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

In [ ]:
import logging

# Clean up any previous logs
for log_file in ["logger.log", "web.log", "tunnel.log"]:
    if os.path.exists(log_file):
        os.remove(log_file)
        print(f"🧹 Cleaned up {log_file}")

# Configure logging with DEBUG log level.
logging.basicConfig(
    filename="logger.log",
    level=logging.DEBUG,
    format="%(filename)s:%(lineno)s %(levelname)s:%(message)s",
)

print("✅ Logging configured")

In [ ]:
from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers


# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("✅ Helper functions defined.")

In [ ]:
!adk create research-manager --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY

In [ ]:
%%writefile research-manager/agent.py

from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import google_search, AgentTool
from google.genai import types

retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

research_1 = LlmAgent(
    name="Researcher1",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
    Identify the **first research topic** from user query. Then use `google_search` to find research papers from the following sources:
    
    1. arXiv
    2. ScienceDirect
    3. Google Scholar
    
    From every research paper found, do the following:
    
    1. Calculate a semantic similarity score between the topic and the research paper title. If it is EQUAL TO or GREATER than 0.5, proceed to step 2.
    2. Read 'Abstract', 'Introduction', and 'Conclusion' sections. Update the semantic similarity score. Proceed to step 3.
    3. If the score is EQUAL TO or GREATER than 0.5, store the research paper title temporarily. Proceed to step 4.
    4. Repeat steps 1, 2, and 3 for multiple research papers. Proceed to step 5. Proceed to step 6.
    5. Keep ONLY 3 research papers with the highest similarity scores. DISCARD the rest.
    6. Present key findings from these research papers.
    7. URLs to access the research papers from step 5 MUST be presented.
    
    Include citations.""",
    tools=[google_search]
)

research_2 = LlmAgent(
    name="Researcher2",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""Identify the **second research topic** from the user query. Then use `google_search` to find research papers from the following sources:
    
    1. arXiv
    2. ScienceDirect
    3. Google Scholar
    
    From every research paper found, do the following:
    
    1. Calculate a semantic similarity score between the topic and the research paper title. If it is EQUAL TO or GREATER than 0.5, proceed to step 2.
    2. Read 'Abstract', 'Introduction', and 'Conclusion' sections. Update the semantic similarity score. Proceed to step 3.
    3. If the score is EQUAL TO or GREATER than 0.5, store the research paper title temporarily. Proceed to step 4.
    4. Repeat steps 1, 2, and 3 for multiple research papers. Proceed to step 5. Proceed to step 6.
    5. Keep ONLY 3 research papers with the highest similarity scores. DISCARD the rest.
    6. Present key findings from these research papers.
    7. URLs to access the research papers from step 5 MUST be presented.
    
    Include citations.""",
    tools=[google_search]
)

research_3 = LlmAgent(
    name="Researcher3",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""Identify the **third research topic** from the user query. Then use `google_search` to find research papers from the following sources:
    
    1. arXiv
    2. ScienceDirect
    3. Google Scholar
    
    From every research paper found, do the following:

    1. Calculate a semantic similarity score between the topic and the research paper title. If it is EQUAL TO or GREATER than 0.5, proceed to step 2.
    2. Read 'Abstract', 'Introduction', and 'Conclusion' sections. Update the semantic similarity score. Proceed to step 3.
    3. If the score is EQUAL TO or GREATER than 0.5, store the research paper title temporarily. Proceed to step 4.
    4. Repeat steps 1, 2, and 3 for multiple research papers. Proceed to step 5.
    5. Keep ONLY 3 research papers with the highest similarity scores. DISCARD the rest. Proceed to step 6.
    6. Present key findings from these research papers.
    7. URLs to access the research papers from step 5 MUST be presented.
    
    Include citations.""",
    tools=[google_search]
)

# === 2. Summarizers ===
summ_1 = LlmAgent(
    name="Summarizer1",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""Summarize findings from `Researcher1` in NOT more than 200 words.
    AND

    URLs to the research papers from 'Researcher1' MUST be presented.
    """,
    tools=[AgentTool(agent=research_1)]
)

summ_2 = LlmAgent(
    name="Summarizer2",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""Summarize findings from `Researcher2` in NOT more than 200 words.
    AND

    URLs to the research papers from 'Researcher2' MUST be presented.
    """,
    tools=[AgentTool(agent=research_2)]
)

summ_3 = LlmAgent(
    name="Summarizer3",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""Summarize findings from `Researcher3` in NOT more than 200 words.
    AND

    URLs to the research papers from 'Researcher3' MUST be presented.
    """,
    tools=[AgentTool(agent=research_3)]
)

# === 3. Coordinators (Sequential) ===
coord_1 = SequentialAgent(name="Coord1", sub_agents=[research_1, summ_1])
coord_2 = SequentialAgent(name="Coord2", sub_agents=[research_2, summ_2])
coord_3 = SequentialAgent(name="Coord3", sub_agents=[research_3, summ_3])

def general_response() -> str:
    return f"I am a research manager. I cannot respond to general questions."

def my_functionalities() -> str:
    return f"I am a research-based agent. My purpose is to assist you in research-based tasks. I can assist you in three research topics at once."

root_agent = LlmAgent(
    name="ExecutiveResearcher",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""You are a research manager agent. You MUST follow these rules strictly:

    1. If the query contains research topics (requests for research, papers, findings, etc.), use the coordination agents 'Coord1', 'Coord2', 'Coord3' to gather summaries. Then present the summaries and highlight **Common Themes**, **Challenges**, and **Three key-takeaways**.
    2. If the query contains only one research topic, use only 'Coord1'.
    3. If the query asks for research materials (e.g., research papers), you MUST present the links used by coordination agents.
    
    CRITICAL: 
    1. For non-research queries (e.g., greetings, casual questions, or general conversation), you are REQUIRED to call the 'general_response' function AND return EXACTLY what it provides. DO NOT generate your own response.
    2. For queries like 'What can you do?', 'What is your purpose?', 'Tell me about yourself', you are REQUIRED to call the 'my_functionalities' function AND return EXACTLY what it provides. DO NOT generate your own response.
    """,
    tools=[AgentTool(agent=coord_1), AgentTool(agent=coord_2), AgentTool(agent=coord_3), general_response, my_functionalities]
)

root_agent.description = "Assist in conducting research. Research → summaries → executive brief."


In [ ]:
url_prefix = get_adk_proxy_url()

In [ ]:
!adk web --log_level DEBUG --url_prefix {url_prefix}